In [4]:
import logging

logger = logging.getLogger('Log.Parser1')
logger.setLevel(logging.DEBUG)
#formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s',datefmt='%m-%d %H:%M:%S')
formatter = logging.Formatter('%(asctime)s - %(message)s',datefmt='%m-%d %H:%M:%S')
fh1 = logging.FileHandler(filename="nb_mytry.py.log", mode='a')
console = logging.StreamHandler()
console.setFormatter(formatter)
console.setLevel(logging.INFO)
fh1.setLevel(logging.DEBUG)
fh1.setFormatter(formatter)
logger.addHandler(console)
logger.addHandler(fh1)

stock = '8299'

In [26]:
from finlab.data import Data
import pandas as pd
data = Data()
s89 = data.get("上月營收")
#s89.to_csv('allstock.csv')
s89['8299']


date
2017-01-10    3837687.0
2017-02-10    3067342.0
2017-03-10    3147077.0
2017-04-10    2779274.0
2017-05-10    3629516.0
                ...    
2023-02-10    4002055.0
2023-03-10    2878167.0
2023-04-10    3274873.0
2023-05-10    3925381.0
2023-06-10    3366959.0
Name: 8299, Length: 78, dtype: float64

In [ ]:
from finlab.data import Data
import pandas as pd
data = Data()
s89 = data.get("收盤價")
s89.to_csv('allstock.csv')
s0050 = s89['0050']

#s0050.to_csv('00501.csv')
df = pd.DataFrame(s0050)
df =df[ df.index> '2019-12-01']
df = df.rename(columns={'0050': 'Close'})
df


In [ ]:
headers = {'Content-Type': 'application/x-www-form-urlencoded; charset=utf-8'}
res = cr.requests_get('https://goodinfo.tw/tw/StockDividendSchedule.asp?STOCK_ID=0050',headers=headers)

#res = requests.get('https://goodinfo.tw/tw/StockDividendSchedule.asp?STOCK_ID=00878')

res.encoding = 'utf-8'
soup = BeautifulSoup(res.text, "html.parser")
header = soup.find_all("table")
len(header)
header[18]
dddd = pd.read_html(res.text)
ssd = dddd[16]
#df_divtation= df_divtation.rename(columns={df_divtation.columns[0][0]: 'year'})
columns = ['股利發放年度','股利所屬盈餘期間','股東會日期','除息交易日','除息參考價','填息完成日','填息花費日數','現金股利發放日','除權交易日',
'除權參考價','填權完成日','填權花費日數','盈餘','公積','合計','盈餘','公積','合計1','股利合計']
ssd = pd.DataFrame(columns = columns,data= ssd.values)
ssd

In [9]:
#get 00878
import pandas as pd
import yfinance as yf
#stkid = '00878.TW'
#divstkid = '00878'

stkid = '0050.TW'
divstkid = '0050'


def getstkPrice(stkid,start,end):
    df = yf.download(stkid,start=start, end=end)
    #df.rename()
    file = f"{stkid}.pikle"
    df.to_pickle(file)
    return df

def getstkPricefromPickle(file):
    df = pd.read_pickle(file)
    #df.rename()
    return df
#save
#stkid = '00878.TW'
file = f"{stkid}.pikle"
#df = getstkPrice('00878.TW','2000-01-01','2023-06-01')
#df.to_pickle(file)
df =  getstkPricefromPickle(file)

#get div
#divstkid = '00878'
divtationfile = f"{divstkid}_div_ratio.pickle"    
#getMyStkdivration(stkid)  frominternet



df_divtation = pd.read_pickle(divtationfile)
df1 = makeMouthLastDayMark(df)
df1 = makeDivRationDataFromMyPKL(df,df_divtation['div_ratio'])

df1.to_csv('test.csv')

In [2]:
import pandas as pd
import requests
import finlab.crawler as cr
def getMyStkdivration(id):
    headers = {'Content-Type': 'application/x-www-form-urlencoded; charset=utf-8'}
    res = cr.requests_get('https://goodinfo.tw/tw/StockDividendSchedule.asp?STOCK_ID='+id,headers=headers)

    #res = requests.get('https://goodinfo.tw/tw/StockDividendSchedule.asp?STOCK_ID=00878')

    res.encoding = 'utf-8'
    soup = BeautifulSoup(res.text, "html.parser")
    header = soup.find_all("table")
    len(header)
    header[18]
    dddd = pd.read_html(res.text)
    ssd = dddd[16]
    #df_divtation= df_divtation.rename(columns={df_divtation.columns[0][0]: 'year'})
    columns = ['股利發放年度','股利所屬盈餘期間','股東會日期','除息交易日','除息參考價','填息完成日','填息花費日數','現金股利發放日','除權交易日',
    '除權參考價','填權完成日','填權花費日數','盈餘','公積','合計','盈餘','公積','合計1','股利合計']
    ssd = pd.DataFrame(columns = columns,data= ssd.values)
    ssd = ssd[ssd['股利發放年度'].str.len() <= 4]
    new_df = pd.DataFrame({'date': ssd['除息交易日']})
    new_df['div_ratio']=ssd['合計']
    new_df['date'] = pd.to_datetime(new_df['date'], format="%y'%m/%d")
    new_df.set_index('date',drop=True,inplace=True)
    divtationfile = f"{id}_div_ratio.pickle"
    new_df.to_pickle(divtationfile)
    return new_df



In [3]:
def makeMouthLastDayMark(df):
    y=0
    m=0
    df['lastdate'] = 0
    s1 = pd.Series([])
    sid = 0
    for idx in df.index:
        if y==0 and m==0:
            y=idx.year
            m=idx.month
        if y==idx.year and m==idx.month:
            #print(f"{idx} >> {df['0050'][idx]}")
            s1.at[sid] = idx
            sid = sid +1
        else:

            idx1 = s1.max()
            #df['lastdate'][idx1] = 1
            df.loc[idx1,'lastdate'] = 1
            sid = 0
            s1 = pd.Series([])
            y=idx.year
            m=idx.month
    return df
df1 = makeMouthLastDayMark(df)


In [4]:
def makeDivRationDataFromMyPKL(df,ss):
    df1['div_ratio'] = ss
    df1['div_ratio'].fillna(0,inplace=True)
    df1['div_ratio'] = df1['div_ratio'].astype(float)
    return df




In [5]:

def makeDivRationData(df,stkid):
    data = Data()
    s89 = data.get("權值_息值")
    s89 = s89[stkid] 
    ss = s89[s89.notna()]
    #print(ss)
    df1['div_ratio'] = ss
    #df1.index

    df1['div_ratio'].fillna(0,inplace=True)
    return df




In [ ]:
#除權息一次買回
import numpy as np
def StragymakeBuy_divBuyAll_CoseAverageCode(df1):
    df1['stk_cnt']=0
    df1['profit_total']=0
    df1['cost']  = 0
    df1['average_cost']  = 0
    df1['earn_p'] = 0
    df1['pprice'] = 0
    cost = 0
    total = 0
    searn = 0.1
    average_cose = 0
    div = 0
    profit_total = 0
    new_average_buy = 0
    p_idx = 0
    month = 3
    for idx in df1.index:
        earn_p = 0
        if average_cose>0:
            earn_p = (df1['Close'][idx] - average_cose) /  average_cose
        #print(earn_p)
        df1.loc[idx,'earn_p'] = earn_p
        if earn_p >= searn:
             #df1['earn_p'][idx] = earn_p 
             profit_total += df1['Close'][idx]  * total
             df1.loc[idx,'earn_p'] = earn_p
             df1.loc[idx,'profit_total'] = profit_total
             new_average_buy = profit_total/month
             total = 0 
             average_cose = 0
             div = 0
             earn_p =0
             cost = 0
        if df1['div_ratio'][idx] !=0:
            pprice = df1['Close'][p_idx]
            df1.loc[idx , 'pprice'] = pprice
            if (profit_total) > 0: #除權息一次買回
                total += profit_total/pprice
                cost += profit_total
            div = total * df1['div_ratio'][idx]
            sstk = div/df1['Close'][idx] 
            cost += div
            total += sstk
            average_cose = cost/total
            profit_total = 0
        else:
            p_idx = idx
            #df1.loc[idx,'stk_cnt']=total
            #df1.loc[idx,'average_cost']  = 
        if (df1['lastdate'][idx] == 1):
            total += 1000
            cost += (df1['Close'][idx] * 1000)
            if profit_total > 0:
                sstk = new_average_buy / df1['Close'][idx]
                total += sstk #total stk cnt
                cost += new_average_buy
                profit_total -= new_average_buy
                df1.loc[idx,'profit_total'] = profit_total
            average_cose = cost/total


        df1.loc[idx,'cost'] = div
        df1.loc[idx,'stk_cnt'] = total
        #df1.loc[idx,'byPrice'] = df1['Close'][idx]
        df1.loc[idx,'cost'] +=  (df1['Close'][idx] * 1000)
        df1.loc[idx,'average_cost']  = average_cose
    return df1
df2 = StragymakeBuy_divBuyAll_CoseAverageCode(df1)
#df1.dropna(inplace=True)
#df1['earn_p'] = ((df1['Close'] - df1['average_cost']) /df1['average_cost']) * 100
df2.to_csv('00502_buyall_befordiv_10.csv')
df2.tail()

In [ ]:
import numpy as np
#buy all , if meet profit rage, sell all and average each month
def AndyStragymakeBuyCoseAverageCode(df1):
    df1['stk_cnt']=0
    df1['remain_asset']=0
    df1['profit_total']=0
    df1['cost']  = 0
    df1['average_cost']  = 0
    df1['earn_p'] = 0
    cost = 0
    total = 0
    average_cost = 0
    div = 0
    remain_asset = 10000000
    sell_asset = 0
    new_average_buy = 0
    for idx in df1.index:
        earn_p = 0
        if average_cost>0:
            earn_p = (df1['Close'][idx] - average_cost) /  average_cost
        #print(earn_p)
        df1.loc[idx,'earn_p'] = earn_p
        if earn_p >= searn:
             #df1['earn_p'][idx] = earn_p 
             sell_asset += df1['Close'][idx]  * total
             df1.loc[idx,'earn_p'] = earn_p
             df1.loc[idx,'remain_asset'] = remain_asset
             #df1.loc[idx,'sell_asset'] = sell_asset
             new_average_buy = sell_asset/month
             total = 0 
             average_cost = 0
             div = 0
             earn_p =0
             cost = 0
    
        if df1['div_ratio'][idx] !=0:
            div = total * df1['div_ratio'][idx]
            sstk = div/df1['Close'][idx] 
            cost += div
            total += sstk
            average_cost = cost/total
            
        if (df1['lastdate'][idx] == 1):
            if remain_asset > (df1['Close'][idx] * 1000):
                total += 1000
                cost += (df1['Close'][idx] * 1000)
                remain_asset -=(df1['Close'][idx] * 1000)
            if sell_asset > 0: #buy all befor div date
                if sell_asset <= new_average_buy:
                    new_average_buy = sell_asset
                    #total += sell_asset / df1['0050'][idx] 
                    sell_asset = 0
                else:
                    sell_asset -= new_average_buy
                    if sell_asset < 1:
                        sell_asset = 0

                sstk = new_average_buy / df1['Close'][idx]
                total += sstk #total stk cnt
                cost += new_average_buy
            #if sell_asset == 0: #buy all befor div date    
            #    remain_asset-= (df1['0050'][idx] * 1000)
            #    cost += (df1['0050'][idx] * 1000)
            #    total += 1000
                #df1.loc[idx,'remain_asset'] = remain_asset
            average_cost = cost/total
        df1.loc[idx,'sell_asset'] = sell_asset
        df1.loc[idx,'remain_asset'] = remain_asset
        df1.loc[idx,'cost'] = div
        df1.loc[idx,'stk_cnt'] = total
        df1.loc[idx,'profit_total'] = (df1['Close'][idx]*total)+remain_asset+sell_asset
        #print(f"=={df1['Close'][idx]}\{total}\{remain_asset}==")
        df1.loc[idx,'cost'] +=  (df1['Close'][idx] * 1000)
        df1.loc[idx,'average_cost']  = average_cost
    return df1

searn = 0.1
month = 36
df1 = AndyStragymakeBuyCoseAverageCode(df1)
#df1.dropna(inplace=True)
#df1['earn_p'] = ((df1['Close'] - df1['average_cost']) /df1['average_cost']) * 100
df1.to_csv(f"{divstkid}_andy_{searn*100}_{month}_wDiv.csv")
df1.tail()

In [ ]:
import numpy as np
#buy by month
def StragymakeBuy_EachMonth(df1):
    df1['stk_cnt']=0
    df1['remain_asset']=0
    df1['profit_total']=0
    df1['cost']  = 0
    df1['average_cost']  = 0
    df1['earn_p'] = 0
    remain_asset = 10000000
    cost = 0
    total = 0
    average_cost = 0
    div = 0
    buystkCnt =1000
    #remain_asset = 0
    new_average_buy = 0
    for idx in df1.index:
        earn_p = 0
        #if average_cost>0:
        #    earn_p = (df1['Close'][idx] - average_cost) /  average_cost
        #print(earn_p)
        df1.loc[idx,'earn_p'] = earn_p
        #if earn_p >= searn:
             #df1['earn_p'][idx] = earn_p 
        #     remain_asset += df1['Close'][idx]  * total
        #     df1.loc[idx,'earn_p'] = earn_p
        #     df1.loc[idx,'remain_asset'] = remain_asset
        #     new_average_buy = remain_asset/month
        #     total = 0 
        #     average_cost = 0
        #     div = 0
        #     earn_p =0
        #     cost = 0
    
        if df1['div_ratio'][idx] !=0:
            div = total * df1['div_ratio'][idx]
            sstk = div/df1['Close'][idx] 
            cost += div
            total += sstk
            average_cost = cost/total
            
        if (df1['lastdate'][idx] == 1):
            if remain_asset >= (df1['Close'][idx] * buystkCnt):
                total += buystkCnt
                cost += (df1['Close'][idx] * buystkCnt)
                charge = df1['Close'][idx]  * buystkCnt * 0.001425
                #charge += df1['Close'][idx]  * buystkCnt * 0.001

                remain_asset -=(df1['Close'][idx] * buystkCnt)
                remain_asset -=charge




            #if remain_asset > 0: #buy all befor div date
            #    if remain_asset <= new_average_buy:
            #        new_average_buy = remain_asset
            #        remain_asset = 0
            #    else:
            #        remain_asset -= new_average_buy

            #    sstk = new_average_buy / df1['Close'][idx]
            #    total += sstk #total stk cnt
            #    cost += new_average_buy
                
                #df1.loc[idx,'remain_asset'] = remain_asset
            average_cost = cost/total

        df1.loc[idx,'remain_asset'] = remain_asset
        df1.loc[idx,'cost'] = div
        df1.loc[idx,'stk_cnt'] = total
        df1.loc[idx,'profit_total'] = (df1['Close'][idx]*total)+remain_asset
        #print(f"=={df1['Close'][idx]}\{total}\{remain_asset}==")
        df1.loc[idx,'cost'] +=  (df1['Close'][idx] * buystkCnt)
        df1.loc[idx,'average_cost']  = average_cost
    return df1

searn = 0.05
month = 36
df1 = StragymakeBuy_EachMonth(df1)
#df1.dropna(inplace=True)
#df1['earn_p'] = ((df1['Close'] - df1['average_cost']) /df1['average_cost']) * 100
df1.to_csv(f"{divstkid}_{searn*100}_{month}_wDiv.csv")
df1.tail()

,Open,High,Low,Close,Adj Close,Volume,lastdate,div_ratio,stk_cnt,remain_asset,profit_total,cost,average_cost,earn_p
Date,,,,,,,,,,,,,,
2023-05-25,18.000000,18.049999,17.950001,18.049999,18.049999,53639604,0,0.0,37432.751538,9.412494e+06,1.008816e+07,28006.160343,17.283726,0
2023-05-26,18.059999,18.230000,18.059999,18.170000,18.170000,67662452,0,0.0,37432.751538,9.412494e+06,1.009265e+07,28126.161182,17.283726,0
2023-05-29,18.260000,18.410000,18.260000,18.360001,18.360001,70895430,0,0.0,37432.751538,9.412494e+06,1.009976e+07,28316.161716,17.283726,0
2023-05-30,18.379999,18.440001,18.260000,18.340000,18.340000,60513537,0,0.0,37432.751538,9.412494e+06,1.009901e+07,28296.161259,17.283726,0
2023-05-31,18.340000,18.389999,18.280001,18.309999,18.309999,40310301,0,0.0,37432.751538,9.412494e+06,1.009789e+07,28266.160572,17.283726,0


In [ ]:
df1.info()

In [ ]:
#spider web sell 1天判斷一次 賣1張

import numpy as np
#buy by month
def StragymakeBuy_spide_web(df1,searn):
    
    df1['stk_cnt']=0
    df1['profit_total']=0
    df1['cost']  = 0
    df1['average_cost']  = 0
    df1['earn_p'] = 0
    df1['remain_asset']=0
    cost = 0
    total = 0
    average_cost = 0
    div = 0
    remain_asset = 10000000
    buystkCnt = 1000
    new_average_buy = 0
    my_array = np.array([])
    for idx in df1.index:
        earn_p = 0
        if len(my_array) >0:
            #earn_p = (df1['0050'][idx] - my_array.min()) /  my_array.min()
            earn_p = (df1['Close'][idx] - my_array.mean()) /  my_array.mean()
            df1.loc[idx,'earn_p'] = earn_p
            df1.loc[idx,'mean'] = my_array.mean()
            if idx == idx == pd.to_datetime('2009-01-15',format="%Y-%m-%d"):
                aaa = 0
            
            if earn_p >= searn:
                #df1['earn_p'][idx] = earn_p

                
                minPrice = my_array[0]
                my_array = np.delete(my_array, 0)
                my_array = np.sort(my_array)
                if len(my_array) >0:
                    average_cost = my_array.mean() * buystkCnt
                else:
                    average_cost = 0
                charge = df1['Close'][idx]  * buystkCnt * 0.001425
                charge += df1['Close'][idx]  * buystkCnt * 0.001

                remain_asset += ((df1['Close'][idx]  * buystkCnt) - (charge))
                cost-=(minPrice*buystkCnt)
                total-=buystkCnt
                df1.loc[idx,'earn_p'] = earn_p
                df1.loc[idx,'remain_asset'] = remain_asset

            if earn_p < (searn *-1):
                if remain_asset >= (df1['Close'][idx] * buystkCnt):
                    total += buystkCnt
                    cost += (df1['Close'][idx] * buystkCnt)
                    my_array = np.append(my_array,df1['Close'][idx])
                    my_array = np.sort(my_array)
                    average_cost = my_array.mean()*buystkCnt
                    #new_average_buy =  (df1['0050'][idx] * 1000)
                    #if remain_asset >= (df1['Close'][idx] * buystkCnt):
                    sstk = buystkCnt
                    remain_asset -=  (df1['Close'][idx] * buystkCnt)

                    charge = df1['Close'][idx]  * buystkCnt * 0.001425
                    
                    remain_asset -= charge
                    #charge += df1['Close'][idx]  * 1000 * 0.001                    
                    #total += buystkCnt #total stk cnt
                    df1.loc[idx,'remain_asset'] = remain_asset    
            
            if df1['div_ratio'][idx] !=0:
                div = total * df1['div_ratio'][idx]
                sstk = div/df1['Close'][idx] 
                #cost += div
                total += sstk
                #average_cost = cost/total
                
            if (df1['lastdate'][idx] == 1):
                if total<buystkCnt: #if less then 1000, buy 1000
                    total += buystkCnt
                    cost += (df1['Close'][idx] * buystkCnt)
                    my_array = np.append(my_array,df1['Close'][idx])
                    my_array = np.sort(my_array,-1)
                    average_cost = my_array.mean()*buystkCnt
                    remain_asset -= (df1['Close'][idx] * buystkCnt)

                    charge = df1['Close'][idx]  * buystkCnt * 0.001425
                    remain_asset -= charge
                    #charge += df1['Close'][idx]  * buystkCnt * 0.001

                #if remain_asset > 0: #buy all befor div date
                #    if remain_asset <= new_average_buy:
                #        new_average_buy = remain_asset
                #        remain_asset = 0
                #    else:
                #        remain_asset -= new_average_buy

                #    sstk = new_average_buy / df1['0050'][idx]
                #    total += sstk #total stk cnt
                #    cost += new_average_buy
                    
                    #df1.loc[idx,'remain_asset'] = remain_asset
            #   average_cost = cost/total
        else:
            if (df1['lastdate'][idx] == 1):
                if len(my_array) ==0: #if less then buystkCnt, buy buystkCnt
                    total += buystkCnt
                    cost += (df1['Close'][idx] * buystkCnt)
                    my_array = np.append(my_array,(df1['Close'][idx]))
                    my_array = np.sort(my_array,-1)
                    average_cost = my_array.mean()*buystkCnt 
                    charge = df1['Close'][idx]  * buystkCnt * 0.001425
                    remain_asset -= (df1['Close'][idx] * buystkCnt)
                    remain_asset -= charge
        df1.loc[idx,'earn_p'] = earn_p                    
        df1.loc[idx,'remain_asset'] = remain_asset
        df1.loc[idx,'cost'] = div
        df1.loc[idx,'stk_cnt'] = total
        df1.loc[idx,'profit_total'] = (df1['Close'][idx]*total)+remain_asset
        #print(f"=={df1['Close'][idx]}\{total}\{remain_asset}==")
        df1.loc[idx,'cost'] +=  (df1['Close'][idx] * buystkCnt)
        df1.loc[idx,'average_cost']  = average_cost
    return df1

searn = 0.03
month = 36
df2 = StragymakeBuy_spide_web(df1,searn)
#df1.dropna(inplace=True)
#df1['earn_p'] = ((df1['Close'] - df1['average_cost']) /df1['average_cost']) * 100
df2.to_csv(f"{divstkid}_{searn*100}_spide_web_mean_charge.csv")
df2.tail()


In [20]:
#spider web , use price split sell/buy

import numpy as np
#buy by month
def splitNet(price,count):
    d = price / count
    webnet = []
    for i in range(0,count):
        webnet.append(price + (i*d))
    for i in range(0,count):
        webnet.append(price - (i*d))
    webnet.sort()
    return webnet
def findUP_LOWPrice(webnet,price):
    up=0
    low = 0
    for i in range(0,len(webnet)):
        if price < webnet[i]:
            up = webnet[i]
            low = webnet[i-1]
            break
    return up,low


def StragymakeBuy_spide_web_splitPrice(df1,searn,webnet):
    
    df1['stk_cnt']=0
    df1['profit_total']=0
    df1['cost']  = 0
    df1['average_cost']  = 0
    df1['earn_p'] = 0
    df1['remain_asset']=0
    df1['totalPay'] =0
    df1['totalEarn'] =0
    cost = 0
    total = 0
    average_cost = 0
    div = 0
    remain_asset = 10000000
    totalPay = 0
    buystkCnt = 1000
    new_average_buy = 0
    up=low = 0
    my_array = np.array([])


    for idx in df1.index:
        earn_p = 0
        if len(my_array) >0:
            #earn_p = (df1['0050'][idx] - my_array.min()) /  my_array.min()
            #earn_p = (df1['Close'][idx] - my_array.mean()) /  my_array.mean()
            cp = df1['Close'][idx]
            if up ==0:
                up,low = findUP_LOWPrice(webnet,df1['Close'][idx])
            
            df1.loc[idx,'earn_p'] = earn_p
            df1.loc[idx,'mean'] = my_array.mean()
            if idx == pd.to_datetime('2009-01-15',format="%Y-%m-%d"):
                aaa = 0
            
            if cp > up and len(my_array) >0:
                minPrice = my_array[0]
                
                my_array = np.delete(my_array, 0)
                my_array = np.sort(my_array)
                if len(my_array) >0:
                    average_cost = my_array.mean() * buystkCnt
                else:
                    average_cost = 0
                charge = df1['Close'][idx]  * buystkCnt * 0.001425
                charge += df1['Close'][idx]  * buystkCnt * 0.001
                totalPay -= (minPrice*buystkCnt-charge)
                remain_asset += ((df1['Close'][idx]  * buystkCnt) - (charge))
                cost-=(minPrice*buystkCnt)
                total-=buystkCnt
                df1.loc[idx,'earn_p'] = earn_p
                df1.loc[idx,'remain_asset'] = remain_asset

            if cp < low:
                if remain_asset >= (df1['Close'][idx] * buystkCnt):
                    up,low = findUP_LOWPrice(webnet,df1['Close'][idx])
                    total += buystkCnt
                    cost += (df1['Close'][idx] * buystkCnt)
                    my_array = np.append(my_array,df1['Close'][idx])
                    my_array = np.sort(my_array)
                    average_cost = my_array.mean()*buystkCnt
                    #new_average_buy =  (df1['0050'][idx] * 1000)
                    #if remain_asset >= (df1['Close'][idx] * buystkCnt):
                    sstk = buystkCnt
                    remain_asset -=  (df1['Close'][idx] * buystkCnt)
                    charge = df1['Close'][idx]  * buystkCnt * 0.001425
                    remain_asset -= charge
                    totalPay += ((df1['Close'][idx] * buystkCnt)+charge)
                    #charge += df1['Close'][idx]  * 1000 * 0.001                    
                    #total += buystkCnt #total stk cnt
                    df1.loc[idx,'remain_asset'] = remain_asset    

            up,low = findUP_LOWPrice(webnet,df1['Close'][idx])
            if df1['div_ratio'][idx] !=0:
                div = total * df1['div_ratio'][idx]
                sstk = div/df1['Close'][idx] 
                #cost += div
                total += sstk
                #average_cost = cost/total
                
            #if (df1['lastdate'][idx] == 1):
            #    if total<buystkCnt: #if less then 1000, buy 1000
            #        total += buystkCnt
            #        cost += (df1['Close'][idx] * buystkCnt)
            #        my_array = np.append(my_array,df1['Close'][idx])
            #        my_array = np.sort(my_array,-1)
            #        average_cost = my_array.mean()*buystkCnt
            #        remain_asset -= (df1['Close'][idx] * buystkCnt)

            #        charge = df1['Close'][idx]  * buystkCnt * 0.001425
            #        remain_asset -= charge
                    #charge += df1['Close'][idx]  * buystkCnt * 0.001

                #if remain_asset > 0: #buy all befor div date
                #    if remain_asset <= new_average_buy:
                #        new_average_buy = remain_asset
                #        remain_asset = 0
                #    else:
                #        remain_asset -= new_average_buy

                #    sstk = new_average_buy / df1['0050'][idx]
                #    total += sstk #total stk cnt
                #    cost += new_average_buy
                    
                    #df1.loc[idx,'remain_asset'] = remain_asset
            #   average_cost = cost/total
        else:
            if len(my_array) ==0: #if less then buystkCnt, buy buystkCnt
                total += buystkCnt
                cost += (df1['Close'][idx] * buystkCnt)
                my_array = np.append(my_array,(df1['Close'][idx]))
                my_array = np.sort(my_array,-1)
                average_cost = my_array.mean()*buystkCnt 
                charge = df1['Close'][idx]  * buystkCnt * 0.001425
                totalPay += ((df1['Close'][idx] * buystkCnt)+charge)
                remain_asset -= (df1['Close'][idx] * buystkCnt)
                remain_asset -= charge
        df1.loc[idx,'earn_p'] = earn_p                    
        df1.loc[idx,'remain_asset'] = remain_asset
        df1.loc[idx,'cost'] = div
        df1.loc[idx,'stk_cnt'] = total
        df1.loc[idx,'profit_total'] = (df1['Close'][idx]*total)+remain_asset
        #print(f"=={df1['Close'][idx]}\{total}\{remain_asset}==")
        df1.loc[idx,'cost'] +=  (df1['Close'][idx] * buystkCnt)
        df1.loc[idx,'average_cost']  = average_cost
        df1.loc[idx,'totalPay'] = totalPay
    return df1

split=50
searn = 0.08
wn = splitNet(120,split)
df2 = StragymakeBuy_spide_web_splitPrice(df1,searn,wn)
df2.to_csv(f"{divstkid}_{split}_spide_web_splitPrice_wCharge.csv")
df2.tail()



,Open,High,Low,Close,Adj Close,Volume,lastdate,div_ratio,stk_cnt,profit_total,cost,average_cost,earn_p,remain_asset,totalPay,totalEarn,mean
Date,,,,,,,,,,,,,,,,,
2023-05-25,123.000000,123.400002,122.449997,123.050003,123.050003,11442859,0,0.0,16633.767166,1.110107e+07,167931.015292,132616.667430,0,9.054280e+06,1.745780e+06,0,131.642308
2023-05-26,125.000000,126.050003,125.000000,125.949997,125.949997,23780520,0,0.0,15633.767166,1.114900e+07,170831.009188,133686.364607,0,9.179925e+06,1.625236e+06,0,132.616667
2023-05-29,127.449997,127.550003,126.500000,126.750000,126.750000,14498868,0,0.0,15633.767166,1.116150e+07,171631.012240,133686.364607,0,9.179925e+06,1.625236e+06,0,133.686365
2023-05-30,126.750000,127.050003,126.449997,126.750000,126.750000,12123804,0,0.0,15633.767166,1.116150e+07,171631.012240,133686.364607,0,9.179925e+06,1.625236e+06,0,133.686365
2023-05-31,126.699997,126.699997,125.150002,126.150002,126.150002,8865297,0,0.0,15633.767166,1.115212e+07,171031.013766,133686.364607,0,9.179925e+06,1.625236e+06,0,133.686365


In [21]:
df2['remain_asset'].min()

8149023.8898119265

In [32]:

#每個月結算
def StragymakeBuy_spide_web_splitPrice(df1,searn,webnet):
    
    df1['stk_cnt']=0
    df1['profit_total']=0
    df1['cost']  = 0
    df1['average_cost']  = 0
    df1['earn_p'] = 0
    df1['remain_asset']=0
    df1['totalPay'] =0
    df1['totalEarn'] =0
    cost = 0
    total = 0
    average_cost = 0
    div = 0
    remain_asset = 10000000
    ori_asset = remain_asset
    totalPay = 0
    buystkCnt = 1000
    new_average_buy = 0
    up=low = 0
    my_array = np.array([])


    for idx in df1.index:
        earn_p = 0
        if len(my_array) >0:
            #earn_p = (df1['0050'][idx] - my_array.min()) /  my_array.min()
            #earn_p = (df1['Close'][idx] - my_array.mean()) /  my_array.mean()
            cp = df1['Close'][idx]
            if up ==0:
                up,low = findUP_LOWPrice(webnet,df1['Close'][idx])
            
            df1.loc[idx,'earn_p'] = earn_p
            df1.loc[idx,'mean'] = my_array.mean()
            if idx == pd.to_datetime('2009-01-15',format="%Y-%m-%d"):
                aaa = 0
            
            if cp > up and len(my_array) >0:
                minPrice = my_array[0]
                
                my_array = np.delete(my_array, 0)
                my_array = np.sort(my_array)
                if len(my_array) >0:
                    average_cost = my_array.mean() * buystkCnt
                else:
                    average_cost = 0
                charge = df1['Close'][idx]  * buystkCnt * 0.001425
                charge += df1['Close'][idx]  * buystkCnt * 0.001
                totalPay -= (minPrice*buystkCnt-charge)
                remain_asset += ((df1['Close'][idx]  * buystkCnt) - (charge))
                cost-=(minPrice*buystkCnt)
                total-=buystkCnt
                df1.loc[idx,'earn_p'] = earn_p
                df1.loc[idx,'remain_asset'] = remain_asset
                #if len(my_array) == 0:


            if cp < low:
                if remain_asset >= (df1['Close'][idx] * buystkCnt):
                    up,low = findUP_LOWPrice(webnet,df1['Close'][idx])
                    total += buystkCnt
                    cost += (df1['Close'][idx] * buystkCnt)
                    my_array = np.append(my_array,df1['Close'][idx])
                    my_array = np.sort(my_array)
                    average_cost = my_array.mean()*buystkCnt
                    #new_average_buy =  (df1['0050'][idx] * 1000)
                    #if remain_asset >= (df1['Close'][idx] * buystkCnt):
                    sstk = buystkCnt
                    remain_asset -=  (df1['Close'][idx] * buystkCnt)
                    charge = df1['Close'][idx]  * buystkCnt * 0.001425
                    remain_asset -= charge
                    totalPay += ((df1['Close'][idx] * buystkCnt)+charge)
                    #charge += df1['Close'][idx]  * 1000 * 0.001                    
                    #total += buystkCnt #total stk cnt
                    df1.loc[idx,'remain_asset'] = remain_asset    

            up,low = findUP_LOWPrice(webnet,df1['Close'][idx])
            if df1['div_ratio'][idx] !=0:
                div = total * df1['div_ratio'][idx]
                sstk = div/df1['Close'][idx] 
                #cost += div
                total += sstk
                #average_cost = cost/total
                
            if (df1['lastdate'][idx] == 1):

                charge = df1['Close'][idx]  * buystkCnt * 0.001425
                charge += df1['Close'][idx]  * buystkCnt * 0.001
            
                if ((df1['Close'][idx]*total)+remain_asset) > (ori_asset+charge):
                    #sell all
                    my_array = np.delete(my_array, np.s_[::])
                    df1.loc[idx,'totalEarn']= ((df1['Close'][idx]*total)+remain_asset) - (ori_asset+charge)
                    remain_asset = ori_asset
                    total = 0

            charge = df1['Close'][idx]  * buystkCnt * 0.001425
            charge += df1['Close'][idx]  * buystkCnt * 0.001  
            #218731                  
            if ((df1['Close'][idx]*total)+remain_asset) - (ori_asset+charge) > 2000:
                if ((df1['Close'][idx]*total)+remain_asset) > (ori_asset+charge):
                    #sell all
                    my_array = np.delete(my_array, np.s_[::])
                    df1.loc[idx,'totalEarn']= ((df1['Close'][idx]*total)+remain_asset) - (ori_asset+charge)
                    remain_asset = ori_asset
                    total = 0
        else:
            if len(my_array) ==0: #if less then buystkCnt, buy buystkCnt
                total += buystkCnt
                cost += (df1['Close'][idx] * buystkCnt)
                my_array = np.append(my_array,(df1['Close'][idx]))
                my_array = np.sort(my_array,-1)
                average_cost = my_array.mean()*buystkCnt 
                charge = df1['Close'][idx]  * buystkCnt * 0.001425
                totalPay += ((df1['Close'][idx] * buystkCnt)+charge)
                remain_asset -= (df1['Close'][idx] * buystkCnt)
                remain_asset -= charge
        df1.loc[idx,'earn_p'] = earn_p                    
        df1.loc[idx,'remain_asset'] = remain_asset
        df1.loc[idx,'cost'] = div
        df1.loc[idx,'stk_cnt'] = total
        df1.loc[idx,'profit_total'] = (df1['Close'][idx]*total)+remain_asset
        #print(f"=={df1['Close'][idx]}\{total}\{remain_asset}==")
        df1.loc[idx,'cost'] +=  (df1['Close'][idx] * buystkCnt)
        df1.loc[idx,'average_cost']  = average_cost
        df1.loc[idx,'totalPay'] = totalPay
    return df1

split=100
searn = 0.06
wn = splitNet(18,split)
df2 = StragymakeBuy_spide_web_splitPrice(df1,searn,wn)
df2.to_csv(f"{divstkid}_{split}_spide_web_splitPrice_wCharge_o2000.csv")
df2.tail()


,Open,High,Low,Close,Adj Close,Volume,lastdate,div_ratio,stk_cnt,profit_total,cost,average_cost,earn_p,remain_asset,totalPay,totalEarn,mean
Date,,,,,,,,,,,,,,,,,
2023-05-25,18.000000,18.049999,17.950001,18.049999,18.049999,53639604,0,0.0,1015.134529,1.000055e+07,18319.999237,18020.000458,0,9.982226e+06,442015.320319,0.0,18.00
2023-05-26,18.059999,18.230000,18.059999,18.170000,18.170000,67662452,0,0.0,1015.134529,1.000067e+07,18440.000076,18020.000458,0,9.982226e+06,442015.320319,0.0,18.02
2023-05-29,18.260000,18.410000,18.260000,18.360001,18.360001,70895430,0,0.0,15.134529,1.000082e+07,18630.000610,0.000000,0,1.000054e+07,424039.842862,0.0,18.02
2023-05-30,18.379999,18.440001,18.260000,18.340000,18.340000,60513537,0,0.0,1015.134529,1.000079e+07,18610.000153,18340.000153,0,9.982175e+06,442405.977515,0.0,19.01
2023-05-31,18.340000,18.389999,18.280001,18.309999,18.309999,40310301,0,0.0,2015.134529,1.000074e+07,18579.999466,18324.999809,0,9.963839e+06,460742.068730,0.0,18.34


In [ ]:
val = []
for split in range(50,120,2):
    wn = splitNet(18.3,split)
    df2 = StragymakeBuy_spide_web_splitPrice(df1,searn,wn)
    res = f"final res = split = {split} - profit = {df2['profit_total'][-1]}"
    val.append(df2['profit_total'][-1])
    print(res)


In [ ]:
max(val)

316474.1070771856

In [ ]:
#spider web  , stop earn ,sell all

import numpy as np
#buy by month
def StragymakeBuy_spide_web_stop_earn_sell_all(df1,searn):
    
    df1['stk_cnt']=0
    df1['profit_total']=0
    df1['cost']  = 0
    df1['average_cost']  = 0
    df1['earn_p'] = 0
    df1['remain_asset']=0
    cost = 0
    total = 0
    average_cost = 0
    div = 0
    remain_asset = 10000000
    new_average_buy = 0
    my_array = np.array([])
    for idx in df1.index:
        earn_p = 0
        if len(my_array) >0:
            #earn_p = (df1['Close'][idx] - my_array.min()) /  my_array.min()
            earn_p = (df1['Close'][idx] - my_array.mean()) /  my_array.mean()
            df1.loc[idx,'earn_p'] = earn_p
            df1.loc[idx,'mean'] = my_array.mean()
            if earn_p >= searn:
                #df1['earn_p'][idx] = earn_p

                remain_asset += df1['Close'][idx]  * total
                my_array = np.delete(my_array, np.s_[:])
                df1.loc[idx,'earn_p'] = earn_p
                df1.loc[idx,'remain_asset'] = remain_asset
                total = 0

            if earn_p < (searn *-1):
                total += 1000
                cost += (df1['Close'][idx] * 1000)
                my_array = np.append(my_array,df1['Close'][idx])
                my_array = np.sort(my_array)
                average_cost = my_array.mean()*1000
                #new_average_buy =  (df1['Close'][idx] * 1000)
                if remain_asset >= (df1['Close'][idx] * 1000):
                    sstk = 1000
                    remain_asset -=  (df1['Close'][idx] * 1000)
                #total += 1000 #total stk cnt
                df1.loc[idx,'remain_asset'] = remain_asset    
            
            #if df1['div_ratio'][idx] !=0:
            #    div = total * df1['div_ratio'][idx]
            #    sstk = div/df1['Close'][idx] 
            #    cost += div
            #    total += sstk
            #    average_cost = cost/total
                
            if (df1['lastdate'][idx] == 1):
                if total<1000: #if less then 1000, buy 1000
                    total += 1000
                    cost += (df1['Close'][idx] * 1000)
                    my_array = np.append(my_array,df1['Close'][idx])
                    my_array = np.sort(my_array,-1)
                    average_cost = my_array.mean()*1000
                    remain_asset -= (df1['Close'][idx] * 1000)
                #if remain_asset > 0: #buy all befor div date
                #    if remain_asset <= new_average_buy:
                #        new_average_buy = remain_asset
                #        remain_asset = 0
                #    else:
                #        remain_asset -= new_average_buy

                #    sstk = new_average_buy / df1['Close'][idx]
                #    total += sstk #total stk cnt
                #    cost += new_average_buy
                    
                    #df1.loc[idx,'remain_asset'] = remain_asset
            #   average_cost = cost/total
        else:
            if (df1['lastdate'][idx] == 1):
                if total<1000: #if less then 1000, buy 1000
                    total += 1000
                    cost += (df1['Close'][idx] * 1000)
                    my_array = np.append(my_array,(df1['Close'][idx]))
                    my_array = np.sort(my_array,-1)
                    average_cost = my_array.mean()*1000 
                    remain_asset -= (df1['Close'][idx] * 1000)
        df1.loc[idx,'remain_asset'] = remain_asset
        df1.loc[idx,'cost'] = div
        df1.loc[idx,'stk_cnt'] = total
        df1.loc[idx,'profit_total'] = (df1['Close'][idx]*total)+remain_asset
        #print(f"=={df1['Close'][idx]}\{total}\{remain_asset}==")
        df1.loc[idx,'cost'] +=  (df1['Close'][idx] * 1000)
        df1.loc[idx,'average_cost']  = average_cost
    return df1

searn = 0.03
month = 36
df2 = StragymakeBuy_spide_web_stop_earn_sell_all(df1,searn)
#df1.dropna(inplace=True)
#df1['earn_p'] = ((df1['Close'] - df1['average_cost']) /df1['average_cost']) * 100
df2.to_csv(f"0050_{searn*100}_spide_web_mean_stop_earn_sell_all.csv")
df2.tail()

,0050,lastdate,div_ratio,stk_cnt,remain_asset,profit_total,cost,average_cost,earn_p,mean,sell_asset
date,,,,,,,,,,,
2023-05-26,125.95,0,0.0,0,10728050,10728050,125950.0,119100.0,0.000000,118.300000,0.0
2023-05-29,126.75,0,0.0,0,10728050,10728050,126750.0,119100.0,0.000000,117.461111,0.0
2023-05-30,126.75,0,0.0,0,10728050,10728050,126750.0,119100.0,0.000000,117.792308,0.0
2023-05-31,126.15,1,0.0,1000,10601900,10728050,126150.0,126150.0,0.000000,118.122000,0.0
2023-06-01,125.15,0,0.0,1000,10601900,10727050,125150.0,126150.0,-0.007927,126.150000,0.0


In [ ]:
import numpy as np
def makeBuyCoseAverageCode(df1):
    df1['stk_cnt']=0
    df1['byPrice']=0
    df1['cost']  = 0
    df1['average_cost']  = 0
    df1['earn_p'] = 0


In [ ]:
data = Data()
s89 = data.get("權值_息值")
s89['0050'] 


date
2020-01-31    2.9
2020-02-13    NaN
2020-02-19    NaN
2020-02-20    NaN
2020-03-04    NaN
             ... 
2023-05-25    NaN
2023-05-26    NaN
2023-05-29    NaN
2023-05-30    NaN
2023-05-31    NaN
Name: 0050, Length: 466, dtype: float64

In [ ]:
data = Data()
s89 = data.get("權值_息值")
s89 = s89['0050'] 

ss = s89[s89.notna()]
ss


date
2020-01-31    2.90
2020-07-21    0.70
2021-01-22    3.05
2021-07-21    0.35
2022-01-21    3.20
2022-07-18    1.80
2023-01-30    2.60
Name: 0050, dtype: float64

In [ ]:
import pandas as pd
df = pd.read_pickle('D:\\mygit\\stockparser\\history\\date_range.pickle')

date_str = '2020-01-28 12:00:00'

# 将时间日期字符串转换为时间戳
timestamp = pd.Timestamp(date_str)

#df['price'][0] = timestamp
type(df['price'])

tuple

In [ ]:
import portalocker
import time
print('ttt')
f = open("testlock.txt",'w')
portalocker.lock(f, portalocker.LOCK_EX)
f.write('test111')
time.sleep(30)
portalocker.unlock(f)



ttt


In [ ]:
import sqlite3
conn = sqlite3.connect('./sqllite3/brokder.db')
cursor = conn.cursor()
print('test')

test


In [ ]:
import pandas as pd


def insertdb(df):
    cnt = 0
    for idx in df.index:
        sql=f"insert into broker_buy (brokerName,buy_tick,sell_tick,diff,stockid,mainBroker,subBroker,date) " \
        f"values('{df['券商名稱'][idx]}',{df['買進張數'][idx]},{df['賣出張數'][idx]},{df['差額'][idx]},'{df['stockid'][idx]}','{df['mainBroker'][idx]}','{df['subBroker'][idx]}','{df['date'][idx]}')"
        print(f"{cnt}/{len(df)}")
        try:
            conn.execute(sql)
            
        except Exception as e:
            print(e)
            print(f"exception : '{df['stockid'][idx]}','{df['mainBroker'][idx]}','{df['subBroker'][idx]}','{df['date'][idx]}'")
        cnt = cnt+1
    conn.commit()
#insertdb(df)

In [ ]:
import os
dir_path = r'./broker1/'

# list to store files
res = []

# Iterate directory
for path in os.listdir(dir_path):
    # check if current path is a file
    if os.path.isfile(os.path.join(dir_path, path)):
        if path.find('.csv') >=0:
            #res.append(path)
            df = pd.read_csv(f'./broker1/{path}',encoding='big5')
            insertdb(df)
        
#res

In [ ]:
def log(*args):
    logger1 = logging.getLogger('Log.Parser1')
    logger1.info(' '.join([str(arg) for arg in args]))

In [ ]:
from finlab.data import Data
import gc
def merge_stock_data(brokerdata,stockid,startdate,cumsum=False):
    data = Data()
    s89 = data.get("收盤價")
    close = pd.DataFrame(s89)
    close = close[stockid]
    close = pd.DataFrame(close)
    close = close[close.index > pd.to_datetime(startdate)]
    #dff = brokerdata #pd.DataFrame(brokerdata)
    if (cumsum) == True:
        s= brokerdata['diff'].cumsum()
    else:
        s=brokerdata['diff']
    close['bkdata'] = s

    close['bkdata'] = close['bkdata'].ffill()
    close.fillna(0,inplace=True)
    arr1 = close[stockid]
    arr2 = close['bkdata']

    #correlation = np.corrcoef(arr1, arr2)[0, 1]
    #print("Correlation:", correlation)
    #cnt = len(close)
    #close = None
    s89 = None
    gc.collect()
    return close

In [ ]:
def merge_sell_buy_data(mid,subid,sotckid,startdate):
    cnt = 0
    sql = f"select sum(diff) as diff,date,stockid from broker_buy where mainBroker='{mid}' and stockid='{sotckid}' group by date,stockid "+\
            "order by date"
    fs = pd.read_sql(sql,conn)
    fs['date'] = pd.to_datetime(fs['date']).dt.date
    fs.set_index('date',inplace=True)
    fs.sort_index(inplace=True)
    return fs

In [ ]:
import datetime
import pandas as pd
mainid = '7750'
subid = '7750'
stockid = '8299'
brokerdata = merge_sell_buy_data(mainid,subid,stockid,pd.to_datetime(datetime.date(2022,3,1)))

close = merge_stock_data(brokerdata,stockid,pd.to_datetime(datetime.date(2022,3,1)))
close.header()

In [ ]:
close.tail()

,8299,bkdata
date,,
2023-02-24,351.0,-8.0
2023-03-01,354.5,-8.0
2023-03-02,365.5,-8.0
2023-03-03,362.5,-8.0
2023-03-06,368.0,-8.0


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN
import numpy as np

# 构造数据
n = 100
A = np.sin(np.linspace(0, 10, n))
B = np.cos(np.linspace(0, 10, n))
A = close['8299']
B = close['bkdata']
# 划分数据集
train_size = int(n * 0.7)
train_A, train_B = A[:train_size], B[:train_size]
test_A, test_B = A[train_size:], B[train_size:]

# 将数据转换成RNN的输入格式
def create_dataset(X, y, time_steps=1):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

time_steps = 10
X_train, y_train = create_dataset(train_A, train_B, time_steps)
X_test, y_test = create_dataset(test_A, test_B, time_steps)

# 构建RNN模型
model = Sequential([
    SimpleRNN(10, input_shape=(time_steps, 1)),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

# 训练模型
model.fit(X_train, y_train, epochs=50, batch_size=16)

# 使用模型进行预测
y_pred = model.predict(X_test)

# 可视化预测结果
import matplotlib.pyplot as plt

plt.plot(y_test, label='True')
plt.plot(y_pred, label='Prediction')
plt.legend()
plt.show()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import numpy as np

# 构造数据
n = 100
B = close['8299']
A = close['bkdata']

# 划分数据集
train_size = int(n * 0.7)
train_A, train_B = A[:train_size], B[:train_size]
test_A, test_B = A[train_size:], B[train_size:]

# 将数据转换成MLP的输入格式
def create_dataset(X, y, time_steps=1):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

time_steps = 1
X_train, y_train = create_dataset(train_A, train_B, time_steps)
X_test, y_test = create_dataset(test_A, test_B, time_steps)

# 构建MLP模型
model = Sequential([
    Dense(10, input_shape=(time_steps,)),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

# 训练模型
model.fit(X_train, y_train, epochs=50, batch_size=16)

# 使用模型进行预测
y_pred = model.predict(X_test)

# 可视化预测结果
import matplotlib.pyplot as plt

plt.plot(y_test, label='True')
plt.plot(y_pred, label='Prediction')
plt.legend()
plt.show()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import numpy as np

# 构造数据
n = 100
A = np.sin(np.linspace(0, 10, n))
B = np.cos(np.linspace(0, 10, n))

# 划分数据集
train_size = int(n * 0.7)
train_A, train_B = A[:train_size], B[:train_size]
test_A, test_B = A[train_size:], B[train_size:]

# 将数据转换成MLP的输入格式
def create_dataset(X, y, time_steps=1):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

time_steps = 10
X_train, y_train = create_dataset(train_A, train_B, time_steps)
X_test, y_test = create_dataset(test_A, test_B, time_steps)

# 构建MLP模型
model = Sequential([
    Dense(10, input_shape=(time_steps,)),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

# 训练模型
model.fit(X_train, y_train, epochs=50, batch_size=16)

# 使用模型进行预测
y_pred = model.predict(X_test)

# 可视化预测结果
import matplotlib.pyplot as plt

plt.plot(y_test, label='True')
plt.plot(y_pred, label='Prediction')
plt.legend()
plt.show()

In [ ]:
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import finlab.crawler as cr

headers = cr.generate_random_header()

session = requests.Session()
retry = Retry(connect=3, backoff_factor=0.5)
adapter = HTTPAdapter(max_retries=retry)
session.mount('http://', adapter)
session.mount('https://', adapter)

#Mborkerid = '9800'
#subborkerid = '9800'
#datestr = '2022-08-12'
#r = requests.get(url,headers = headers)
print(" read data start")
url = "https://www.twse.com.tw/brokerService/brokerServiceAudit?showType=list&stkNo=1020"
#url = "https://www.learncodewithmike.com/2021/05/pandas-and-sqlite.html"
#url = "https://www.twse.com.tw/zh/brokerService/brokerServiceAudit"
r = session.get(url,headers = headers,verify=False)
try:
    df = pd.read_html(r.text)
except Exception as e:
    print('ttt')
    print(str(e))


 read data start


c:\Users\danson_tsui\AppData\Local\Programs\Python\Python37\lib\site-packages\urllib3\connectionpool.py:1052: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.twse.com.tw'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,


In [ ]:
df[3]

,證券商代號,證券商名稱,開業日,地址,電話
0,1021,合庫- 台中,1001202,台中市西區民權路91號6樓,04-22255141
1,1022,合庫-台南,1001202,台南市北區成功路48號3樓,06-2260148
2,1023,合庫-高雄,1001202,高雄市大勇路97號5樓,07-5319755
3,1024,合庫-嘉義,1001202,嘉義市國華街279號2樓,05-2220016
4,1025,合庫-基隆,1001202,基隆市仁二路255號3樓,02-24288468
5,1028,合庫 - 彰化,1010516,彰化縣彰化市民生路279號5樓,04-7295528
6,1029,合庫 - 鳳山,1010518,高雄市鳳山區鳳松路3號,07-7905555
7,102A,合庫-新竹,1001202,新竹市東區東門街60號5樓,03-5262711
8,102C,合庫-自強,1001202,台北市中山區南京東路2段85、87號3樓,02-25218815
9,102E,合庫-桃園,1001202,桃園市桃園區中華路12號3樓,03-347-3456


In [ ]:
df[3]

,證券商代號,證券商名稱,開業日,地址,電話
0,1021,合庫- 台中,1001202,台中市西區民權路91號6樓,04-22255141
1,1022,合庫-台南,1001202,台南市北區成功路48號3樓,06-2260148
2,1023,合庫-高雄,1001202,高雄市大勇路97號5樓,07-5319755
3,1024,合庫-嘉義,1001202,嘉義市國華街279號2樓,05-2220016
4,1025,合庫-基隆,1001202,基隆市仁二路255號3樓,02-24288468
5,1028,合庫 - 彰化,1010516,彰化縣彰化市民生路279號5樓,04-7295528
6,1029,合庫 - 鳳山,1010518,高雄市鳳山區鳳松路3號,07-7905555
7,102A,合庫-新竹,1001202,新竹市東區東門街60號5樓,03-5262711
8,102C,合庫-自強,1001202,台北市中山區南京東路2段85、87號3樓,02-25218815
9,102E,合庫-桃園,1001202,桃園市桃園區中華路12號3樓,03-347-3456


In [ ]:
import sqlite3
conn = sqlite3.connect('./sqllite3/brokder.db')
cursor = conn.cursor()

df[0].to_sql('broker_data_main', conn, if_exists='replace', index=False)

In [ ]:
df1=df[0]
for idx in df1.index:
    url = f"https://www.twse.com.tw/brokerService/brokerServiceAudit?showType=list&stkNo={df1['證券商代號'][idx]}"
#url = "https://www.learncodewithmike.com/2021/05/pandas-and-sqlite.html"
#url = "https://www.twse.com.tw/zh/brokerService/brokerServiceAudit"
    r = session.get(url,headers = headers,verify=False)
    try:
        cursor.execute(f"delete from broker_data_detail where mainid='{df1['證券商代號'][idx]}'")
        df = pd.read_html(r.text)
        df = df[3]
        df['mainid'] = df1['證券商代號'][idx]
        df.to_sql('broker_data_detail', conn, if_exists='append', index=False)
    except Exception as e:
        print(str(e))


In [ ]:
url = f"https://www.twse.com.tw/brokerService/brokerServiceAudit?showType=list&stkNo={df1['證券商代號'][0]}"
url

'https://www.twse.com.tw/brokerService/brokerServiceAudit?showType=list&stkNo=1020'

In [ ]:
import pandas as pd
col = '合庫'
df1 = pd.read_pickle('db/'+stock+'.pkl')
print(df1.tail(5))
close_  = df1['收盤']
close_.to_csv('收盤.csv',encoding='utf-16')


               收盤    漲跌     開盤     最高     最低   合庫  合庫-台中  合庫-台南  合庫-高雄  合庫-嘉義  \
日期                                                                              
2021-04-09  484.0  -8.0  494.0  497.0  480.5 -355    -18     -1    -21      0   
2021-04-12  486.0   2.0  491.0  493.5  484.0 -356    -18     -2    -23      0   
2021-04-13  487.0   1.0  489.5  493.0  485.0 -355    -30     -6    -23      0   
2021-04-14  473.5 -13.5  487.0  487.5  451.0 -349    -21     -6    -20      3   
2021-04-15  489.0  15.5  474.0  489.5  468.0 -354    -25     -6    -22      3   

            ...  大慶-台南  盈溢-籬子內  大和國泰  盈溢-楠梓  致和-台北  全泰  元大寶來期貨  犇亞證券  大慶-高雄  \
日期          ...                                                               
2021-04-09  ...      0      -7   344      1      0  -1       4   -52      3   
2021-04-12  ...      0      -7   319      1      0  -1       4   -52      3   
2021-04-13  ...      0      -7   319      1      0  -1       4   -52      3   
2021-04-14  ...      1      -8   309 

In [ ]:
startcal = 0
for col in df1.columns:
        if col == '最低':
            startcal = 1
            continue
        if startcal:        
            d1 = pd.Series(df1[col])
            #d = df1[col]
            d = d1.tail(5)
            d = d.shift(1) - d
            d = d[d>50]
            if len(d) > 0:
                print(col)
                print(d)
#s = df1.tail(5)
#print(s)
#s = s.shift(1) - s
#s
#a = s[s>0]
#a = a.dropna(axis=1)
#a

台灣摩根士丹利
日期
2021-04-14    177.0
Name: 台灣摩根士丹利, dtype: float64
港商野村
日期
2021-04-13    75.0
2021-04-14    74.0
Name: 港商野村, dtype: float64
新加坡商瑞銀
日期
2021-04-14    161.0
Name: 新加坡商瑞銀, dtype: float64
元富
日期
2021-04-15    60.0
Name: 元富, dtype: float64
國票-敦北
日期
2021-04-14    51.0
Name: 國票-敦北, dtype: float64
摩根大通
日期
2021-04-14    233.0
Name: 摩根大通, dtype: float64
玉山
日期
2021-04-14    61.0
Name: 玉山, dtype: float64
國泰
日期
2021-04-14    99.0
Name: 國泰, dtype: float64
國泰-館前
日期
2021-04-15    197.0
Name: 國泰-館前, dtype: float64
凱基-復興
日期
2021-04-14    89.0
Name: 凱基-復興, dtype: float64
華南永昌
日期
2021-04-15    120.0
Name: 華南永昌, dtype: float64
元大
日期
2021-04-14    101.0
Name: 元大, dtype: float64
日盛-木柵
日期
2021-04-12    51.0
Name: 日盛-木柵, dtype: float64


In [ ]:
#print(df1.head())
column = ['name','sum','overbuy','day','win','count']
def stragy_sell(overbuy,d):
    #overbuy = 30
    startcal = 0
    max = 0
    total = 0

    #發現訊號那天開始算d天的收盤價 - 發現訊號那天開始的隔天收盤價買進 
    profit5 =  close_.shift(d*-1) - close_.shift(-1)
    maxlist = pd.DataFrame(columns = column)
    for col in df1.columns:
        if col == '最低':
            startcal = 1
            continue
        if startcal:        
            d1 = pd.Series(df1[col])
            d2 = d1.diff()
            #d2 = d2.dropna()
            k = profit5[d2<overbuy]
            k = k-4 #手續費 滑價
            k = k.dropna()
            p = k[k > 0]
            n = k[k <=0]
            t = k.sum()
            #t =t-(len(k) * 4)
            if t > 0:
                max = t
                #if len(k) >=10 and (len(p)/len(k))>=0.1:
                    #datac.append(col)
                datac=[[col,t,overbuy,d,round(len(p)/len(k),2),len(k)]]

                #datac=[[1,1,1,1,3,1]]
                d1 = pd.DataFrame(datac, columns=column)
                maxlist=maxlist.append(d1)
                #print('------------maxlist---------')
                #print(maxlist.head())
                
                ss = '{:<10},sum:{:},overbuy :{:}, day:{:},win:[{:.2f}%],count:{:}'.format(col,t,overbuy,d,len(p)/len(k),len(k))
                    
                #log(col , 'sum:',t , ',win:{:}-[{:.2f}%]'.format(len(p),len(p)/len(k)),',lose:{:}-[{:.2f}%]'.format(len(n),len(n)/len(k)),',count :' , len(k))
                #log(k)
                #print('n >> ' , len(n))
                #print('count>>' , len(k))
                #print(col , ">>>" , t)
                total += t
    return total,maxlist

In [ ]:

#print(df1.head())
column = ['name','sum','overbuy','day','win','count']
def stragy(overbuy,d,sb):
    #overbuy = 30
    startcal = 0
    max = 0
    total = 0

    #發現訊號那天開始算d天的收盤價 - 發現訊號那天開始的隔天收盤價買進 
    profit5 =  close_.shift(d*-1) - close_.shift(-1)
    maxlist = pd.DataFrame(columns = column)
    for col in df1.columns:
        if col == '最低':
            startcal = 1
            continue
        if startcal:        
            d1 = pd.Series(df1[col])
            d2 = d1.diff()
            #d2 = d2.dropna()
            k = profit5[d2>overbuy]
            k = k-sb #手續費 滑價
            k = k.dropna()
            p = k[k > 0]
            n = k[k <=0]
            t = k.sum()
            #t =t-(len(k) * 4)
            if t > 0:
                max = t
                #if len(k) >=10 and (len(p)/len(k))>=0.1:
                    #datac.append(col)
                datac=[[col,t,overbuy,d,round(len(p)/len(k),2),len(k)]]

                #datac=[[1,1,1,1,3,1]]
                d1 = pd.DataFrame(datac, columns=column)
                maxlist=maxlist.append(d1)
                #print('------------maxlist---------')
                #print(maxlist.head())
                
                ss = '{:<10},sum:{:},overbuy :{:}, day:{:},win:[{:.2f}%],count:{:}'.format(col,t,overbuy,d,len(p)/len(k),len(k))
                    
                #log(col , 'sum:',t , ',win:{:}-[{:.2f}%]'.format(len(p),len(p)/len(k)),',lose:{:}-[{:.2f}%]'.format(len(n),len(n)/len(k)),',count :' , len(k))
                #log(k)
                #print('n >> ' , len(n))
                #print('count>>' , len(k))
                #print(col , ">>>" , t)
                total += t
    return total,maxlist



列出查詢出來的條件的詳細進場日期

In [ ]:
close_  = df1['收盤']
a = close_.index[0].strftime('%Y-%m-%d')
a
#a = close_.index.astype(str)
#a
#print(a[0])


'2021-01-04'

In [ ]:
close_  = df1['收盤']
d = 7
overbuy = 10
profit5 = close_.shift((d)*-1) - close_.shift(-1)
d1 = pd.Series(df1[name])
d2 = d1.diff()
#print(d2)

k = profit5[d2>overbuy]
print(k)
k = k-0.4 #手續費 滑價

#p = k[k > 0]

#p
#startdate = p.first('1D').index[0]

NameError: name 'name' is not defined

In [ ]:

overbuy = -10
d = 7
name = '合庫'#'永豐金-竹北'
#total,m,p = stragy(overbuy,d)

#profit5 =  close_.shift(d*-1) + close_.shift(-1)
close_  = df1['收盤']
#print(close_.head(30))
close_.to_csv(name+'_close.csv',encoding='utf-16')
#發現訊號那天開始算d天 - 發現訊號那天開始的隔天買進 
profit5 = close_.shift((d)*-1) - close_.shift(-1)
#print(profit5.head(30))
profit5.to_csv(name+'_profit5.csv',encoding='utf-16')

d1 = pd.Series(df1[name])
#print(d1)
d2 = d1.diff()
d2.to_csv(name+'overbuy.csv',encoding='utf-16')
#d2 = d2.dropna()
#print(d2)
k = profit5[d2<overbuy]
#print(k)
k = k-0.2 #手續費 滑價
#print(k)
p = k[k > 0]

#取得訊號發生第一筆
startdate = p.first('1D').index[0]
startdate = startdate.strftime('%Y-%m-%d')
print('start=',p)
clos_startbydate = close_
clos_startbydate = clos_startbydate[startdate:]
#從隔天價格開始買, 所以剔除訊號發生那天
clos_startbydate = clos_startbydate[1:]
print (clos_startbydate.head())
earn_percentage = (clos_startbydate - clos_startbydate[0]) *100
earn_percentage = (earn_percentage/clos_startbydate[0])
print(earn_percentage)
#s = close_[p.first('1D').index:]
#print(s)
n = k[k <=0]
t = k.sum()
print(name)
print(p)
p.to_csv(name+'_buy.csv',encoding='utf-16')



start= 日期
2021-01-08     8.8
2021-01-12    10.3
2021-01-13    15.8
2021-01-14    15.8
2021-01-15    11.8
2021-01-19    10.3
2021-01-29    31.3
2021-02-18    17.3
2021-03-16     3.3
Name: 收盤, dtype: float64
日期
2021-01-11    377.0
2021-01-12    381.5
2021-01-13    389.0
2021-01-14    398.0
2021-01-15    387.0
Name: 收盤, dtype: float64
日期
2021-01-11     0.000000
2021-01-12     1.193634
2021-01-13     3.183024
2021-01-14     5.570292
2021-01-15     2.652520
2021-01-18     1.193634
2021-01-19     2.387268
2021-01-20     0.397878
2021-01-21     5.968170
2021-01-22     9.814324
2021-01-25     6.896552
2021-01-26     4.376658
2021-01-27     5.702918
2021-01-28     3.183024
2021-01-29     0.530504
2021-02-01     1.856764
2021-02-02     2.254642
2021-02-03     4.244032
2021-02-04     2.785146
2021-02-05     2.652520
2021-02-17     8.355438
2021-02-18    10.212202
2021-02-19    12.466844
2021-02-22    16.047745
2021-02-23    17.374005
2021-02-24    14.854111
2021-02-25    23.209549
2021-02-26    2

開始模擬

In [ ]:
total ,maxlist= stragy_sell(-200,4)
maxlist

,name,sum,overbuy,day,win,count
0,合庫,5.0,-200,4,1.0,1
0,日盛-頭份,23.5,-200,4,0.5,2
0,宏遠,19.0,-200,4,0.4,5
0,美林,45.5,-200,4,1.0,2
0,新加坡商瑞銀,2.5,-200,4,0.5,2
0,兆豐-台中,1.5,-200,4,1.0,1
0,國泰,10.0,-200,4,1.0,1
0,國泰-館前,9.0,-200,4,0.5,2
0,凱基-復興,22.5,-200,4,1.0,1
0,永豐金,1.5,-200,4,1.0,1


In [ ]:

max = 0
ss = ''
maxtotal = pd.DataFrame(columns = column)
for s in range(30,300,10):
    for d in range(2,30):
        log('overbuy : ' + str(s)+' day :' + str(d))
        total ,maxlist= stragy(s,d,0.4)
        if total > max:
            ss = 'overbuy : ' + str(s)+' day :' + str(d)
            max = total
        log('total:' + str(total)+'\n')
        maxtotal = maxtotal.append(maxlist)
#for i in ran
maxtotal = maxtotal.reset_index(drop=True)
print(maxtotal.head())
maxtotal.to_csv('maxtotal_buy.csv',encoding='utf-16')
#log('----final-------')
#log(ss)
#log('----single best-------')
#for i in range(0,len(maxlist)):
    #log(maxlist[i])

13 - overbuy : 210 day :7
04-14 15:12:15 - total:3.750000000000016

04-14 15:12:15 - overbuy : 210 day :8
04-14 15:12:16 - total:4.700000000000018

04-14 15:12:16 - overbuy : 210 day :9
04-14 15:12:18 - total:7.100000000000013

04-14 15:12:18 - overbuy : 210 day :10
04-14 15:12:19 - total:10.4

04-14 15:12:19 - overbuy : 210 day :11
04-14 15:12:21 - total:14.149999999999993

04-14 15:12:21 - overbuy : 210 day :12
04-14 15:12:22 - total:16.89999999999999

04-14 15:12:22 - overbuy : 210 day :13
04-14 15:12:24 - total:19.149999999999984

04-14 15:12:24 - overbuy : 210 day :14
04-14 15:12:25 - total:26.500000000000007

04-14 15:12:25 - overbuy : 210 day :15
04-14 15:12:27 - total:29.75000000000001

04-14 15:12:27 - overbuy : 210 day :16
04-14 15:12:29 - total:36.50000000000004

04-14 15:12:29 - overbuy : 210 day :17
04-14 15:12:30 - total:38.55000000000004

04-14 15:12:30 - overbuy : 210 day :18
04-14 15:12:32 - total:39.500000000000036

04-14 15:12:32 - overbuy : 210 day :19
04-14 15:12:3